# RAG Agent Pattern
### When the process needs to *know*

RAG dynamically retrieves external data to provide agents with real-world knowledge without retraining. The agent doesn't guess — it searches, retrieves, and grounds its response in actual documents.

Use this pattern when a process step requires an informed answer — not an action, not a transaction, but an accurate response drawn from your organization's knowledge.

### How It Works

```
Query Reception → Agent processes user request
       ↓
Knowledge Retrieval → Search vectors with Nova embeddings
       ↓
Context Enhancement → Augment prompt with retrieved chunks
       ↓
Response Generation → LLM generates fact-grounded response
```

### Architecture

```
User Query → Agent → search_knowledge_base()
                          │
                    Nova Multimodal Embeddings
                          │
                    ChromaDB (vector search)
                          │
                    Relevant chunks
                          │
             Agent ←──────┘
                │
         Reason over chunks → Grounded response
```

### Business Use Case: AnyComp Telecom Employee Onboarding Q&A

New hires ask an agent about policies, benefits, IT setup, and procedures instead of reading a 30-page PDF.

## Install Dependencies

Run this cell first, then **restart the kernel** and continue.

In [ ]:
!pip3 install -q strands-agents strands-agents-tools boto3 typing_extensions>=4.12.0 chromadb pypdf

## Setup: Build the Vector Store

We extract text from the Employee Onboarding Guide PDF, chunk it, and embed each chunk with **Amazon Nova Multimodal Embeddings** (text mode, 1024 dimensions). Embeddings are stored in **ChromaDB** (local, open-source vector database). No infrastructure to provision.

In [ ]:
import boto3, json
import chromadb
from pathlib import Path
from pypdf import PdfReader

REGION = "us-east-1"
EMBED_MODEL = "amazon.nova-2-multimodal-embeddings-v1:0"

# Find the PDF
PDF_PATH = None
for candidate in [
    Path("../common/sample_data/Employee Onboarding Guide.pdf"),
    Path("common/sample_data/Employee Onboarding Guide.pdf"),
    Path("../../agent-orchestration-patterns/common/sample_data/Employee Onboarding Guide.pdf"),
]:
    if candidate.exists():
        PDF_PATH = candidate
        break
assert PDF_PATH, "Employee Onboarding Guide.pdf not found"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

# ── Extract text from PDF (lightweight, pure Python) ─────────────────────────
print("Reading PDF...")
reader = PdfReader(str(PDF_PATH))
pages_text = [page.extract_text() or "" for page in reader.pages]
pages_text = [t.strip() for t in pages_text if t.strip()]
print(f"   {len(pages_text)} pages with text")

# ── Chunk into ~500 char segments ────────────────────────────────────────────
chunks = []
for page_text in pages_text:
    words = page_text.split()
    current = []
    current_len = 0
    for word in words:
        current.append(word)
        current_len += len(word) + 1
        if current_len >= 500:
            chunks.append(" ".join(current))
            current = []
            current_len = 0
    if current:
        chunks.append(" ".join(current))
print(f"   {len(chunks)} chunks")

# ── Embed with Nova Multimodal Embeddings (text mode) ────────────────────────
print("Embedding with Nova Multimodal Embeddings...")

def embed_text(text, purpose="GENERIC_INDEX"):
    """Embed text using Nova Multimodal Embeddings."""
    body = {
        "taskType": "SINGLE_EMBEDDING",
        "singleEmbeddingParams": {
            "embeddingPurpose": purpose,
            "embeddingDimension": 1024,
            "text": {"truncationMode": "END", "value": text},
        },
    }
    resp = bedrock_runtime.invoke_model(
        modelId=EMBED_MODEL, body=json.dumps(body),
        contentType="application/json", accept="application/json",
    )
    return json.loads(resp["body"].read())["embeddings"][0]["embedding"]

embeddings = [embed_text(chunk) for chunk in chunks]
print(f"   ✅ {len(embeddings)} embeddings (dim={len(embeddings[0])})")

# ── Store in ChromaDB ────────────────────────────────────────────────────────
print("Storing in ChromaDB...")
chroma_client = chromadb.Client()
try: chroma_client.delete_collection("onboarding_guide")
except: pass

collection = chroma_client.create_collection(name="onboarding_guide", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[f"chunk-{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks,
)
print(f"   ✅ {collection.count()} chunks indexed")

## Helper: Search Function

This function embeds the query with Titan and searches ChromaDB. All three frameworks use it.

In [ ]:
def search_onboarding_kb(query: str, n_results: int = 5) -> list:
    """Embed query with Nova and search ChromaDB."""
    query_embedding = embed_text(query, purpose="GENERIC_RETRIEVAL")
    results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
    return results["documents"][0] if results["documents"] else []

# Quick test
test = search_onboarding_kb("first day")
print(f"Test: {len(test)} results")
print(f"Preview: {test[0][:200]}...")

## Sample Questions

In [ ]:
QUESTIONS = [
    'What should I do on my first day at AnyComp Telecom?',
    'How do I set up my laptop and get access to internal systems?',
    'What benefits does AnyComp Telecom offer to new employees?',
]

## Implementation 1: Strands Agents

The Strands approach uses a `@tool` function that calls our search helper. The agent decides when to search and reasons over the retrieved chunks.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def search_knowledge_base(query: str) -> str:
    """Search the AnyComp Telecom onboarding knowledge base.

    Args:
        query: The search query about onboarding, policies, or procedures
    """
    chunks = search_onboarding_kb(query, n_results=5)
    if not chunks:
        return "No relevant information found."
    return "\n\n".join(f"[Chunk {i+1}] {c}" for i, c in enumerate(chunks))


strands_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    region_name=REGION,
    max_tokens=2048,
)

strands_agent = Agent(
    model=strands_model,
    system_prompt=(
        "You are an onboarding assistant for AnyComp Telecom. "
        "Use the search_knowledge_base tool to find relevant information before answering. "
        "Always cite which section of the onboarding guide your answer comes from. "
        "If the knowledge base does not contain the answer, say so clearly."
    ),
    tools=[search_knowledge_base],
)

print("=" * 60)
print("STRANDS AGENTS — RAG Agent")
print("=" * 60)
for q in QUESTIONS:
    print(f"\nQ: {q}")
    print(f"A: {strands_agent(q)}")
    print("-" * 40)

In [ ]:
# ChromaDB is in-memory — nothing to tear down.
# The collection is automatically cleaned up when the kernel restarts.

# If you want to explicitly clear it:
try:
    chroma_client.delete_collection("onboarding_guide")
    print("✅ Deleted ChromaDB collection")
except:
    print("Collection already cleared")